# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, specifically for the FAIR² dataset package.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print general information
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Version: {meta.version}")
print(f"Published: {meta.datePublished}")
print(f"Citation: {meta.citeAs}")
print(f"License: {meta.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema.

In [ ]:
# Discover all record sets and their fields by @id
record_sets = list(dataset.record_sets)

print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"  [@id] {rs['@id']} - {rs.get('name', rs['@id'])}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("    Fields and their @id:")
    for f in fields:
        if isinstance(f, dict):
            print(f"      - {f.get('@id', f)}: {f.get('name', '')}")
        else:
            print(f"      - {f}")
    print()

# For illustration, list the first record in each record set
for rs in record_sets:
    print(f"Example record from record_set @id={rs['@id']}:")
    recs = dataset.records(record_set=rs['@id'])
    try:
        print(next(recs))
    except StopIteration:
        print("  (No records found)")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s discovered above.

In [ ]:
# List record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
print("Record Set @ids:", record_set_ids)

# For this example, we use the first record set. Modify this to work with others if needed.
main_record_set_id = record_set_ids[0]

# Load data from all available record sets
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id}: {df.shape[0]} rows, {df.shape[1]} columns")
    else:
        print(f"No records found for {record_set_id}.")

# Show columns in main DataFrame
main_df = dataframes[main_record_set_id]
print("Columns in main record set:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section demonstrates filtering, normalization, and grouping using a numeric field from the table.

In [ ]:
# Select a numeric field for analysis.
# You may need to inspect the columns of `main_df` to choose; here we show how to select.
numeric_candidates = main_df.select_dtypes(include=['float', 'int']).columns.tolist()
print(f"Numeric candidate fields: {numeric_candidates}")
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # Pick the first numeric field
else:
    raise ValueError('No numeric fields found for analysis.')

# Example threshold to filter (customize to data range as appropriate)
threshold = main_df[numeric_field_id].mean()
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} [mean]: {filtered_df.shape[0]} records")
print(filtered_df.head())

# Normalize numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a likely field (e.g. if there is a categorical or sample column)
group_candidates = main_df.select_dtypes(exclude=['float', 'int']).columns.tolist()
if group_candidates:
    group_field = group_candidates[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"Grouped data (mean {numeric_field_id}) by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable non-numeric group fields found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the numeric field
plt.figure(figsize=(6, 4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If grouping, plot boxplot by group
if group_candidates:
    group_field = group_candidates[0]
    plt.figure(figsize=(12,5))
    sns.boxplot(data=main_df, x=group_field, y=numeric_field_id)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² mass spectrometry dataset using the `mlcroissant` library. We loaded the dataset by Croissant schema URL, listed available record sets, loaded tabular data by `@id`, filtered and normalized a numeric field, and visualized data distributions. You can extend this notebook by exploring other record sets, filtering and visualizing more fields, or applying advanced analysis relevant to your research.

For more info, see the [FAIR² dataset on Senscience](https://sen.science/doi/10.71728/senscience.vq4a-28xa).